# 🔬 Unified SegMoTE vs MoE-Segformer Evaluator (BUSI Dataset)

This notebook trains both the SegMoTE foundation model and the lightweight MoE-Segformer baseline on the **Breast Ultrasound Images (BUSI)** dataset for a direct apples-to-apples comparison.


In [ ]:
# 1. Clone Repository and Install Dependencies
%cd /content
!rm -rf vision_tranformer_moe
!git clone https://github.com/toqeer-ahmed/vision_tranformer_moe.git
%cd /content/vision_tranformer_moe
!pip install -r requirements.txt
!pip install kaggle albumentations tensorboard transformers torch torchvision pandas

## 2. Authenticate with Kaggle

In [ ]:
import os
# Set Kaggle API Token
os.environ['KAGGLE_API_TOKEN'] = "KGAT_f92b021c2b42601bd960c76192014a55"
!mkdir -p ~/.kaggle
!echo "KGAT_f92b021c2b42601bd960c76192014a55" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token
print("Kaggle authentication configured!")

In [ ]:
# 3. Download and Prepare the BUSI Dataset
!kaggle datasets download -d aryashah2k/breast-ultrasound-images-dataset
!unzip -q breast-ultrasound-images-dataset.zip -d busi_temp

import os
import shutil
import glob

dest_dir = "data/medical_dataset"
os.makedirs(os.path.join(dest_dir, "images"), exist_ok=True)
os.makedirs(os.path.join(dest_dir, "masks"), exist_ok=True)

image_files = []
for root, dirs, files in os.walk('busi_temp'):
    for file in files:
        if file.lower().endswith('.png'):
            if 'mask' not in file.lower() and 'segmentation' not in file.lower():
                image_files.append(os.path.join(root, file))

mask_files = []
for root, dirs, files in os.walk('busi_temp'):
    for file in files:
        if file.lower().endswith('.png'):
            if 'mask' in file.lower() or 'segmentation' in file.lower():
                mask_files.append(os.path.join(root, file))

mask_dict = {}
for m in mask_files:
    base = os.path.basename(m).replace('_mask_1', '').replace('_mask_2', '').replace('_mask', '').replace('.png', '')
    mask_dict[base] = m

print(f"Found {len(image_files)} training images. Transferring...")
found_pairs = 0
for img_path in image_files:
    base = os.path.basename(img_path).replace('.png', '')
    if base in mask_dict:
        shutil.copy(img_path, os.path.join(dest_dir, 'images', f'{base}.png'))
        shutil.copy(mask_dict[base], os.path.join(dest_dir, 'masks', f'{base}_mask.png'))
        found_pairs += 1

print(f"Successfully paired and formatted {found_pairs} BUSI images for the framework!")

In [ ]:
# 4. Unified Configuration Injection (For Apples-to-Apples Testing)
import yaml

def update_config(path, epochs=15):
    with open(path, 'r') as f:
        config = yaml.safe_load(f)
    config['training']['epochs'] = epochs
    config['dataset']['name'] = 'medical-image-mask'
    config['dataset']['data_dir'] = 'data/medical_dataset'
    config['dataset']['batch_size'] = 2
    with open(path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

update_config("configs/medical_segmentation.yaml", epochs=20)
update_config("configs/moe_segmentation.yaml", epochs=20)

print("Configurations unified! Training set to 20 epochs.")

In [ ]:
# 5. Train SegMoTE (Foundation Model)
!PYTHONPATH=. python training/train_segmote.py --config configs/medical_segmentation.yaml

In [ ]:
# 6. Train MoE-Segformer (Baseline)
!PYTHONPATH=. python training/train_moe.py --config configs/moe_segmentation.yaml

In [ ]:
# 7. Automated Metric Parser & Comparison
import os
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def get_best_metric(log_dir, tag='Metrics/mIoU'):
    best_val = 0.0
    if not os.path.exists(log_dir): return best_val
    for file in os.listdir(log_dir):
        if file.startswith("events.out.tfevents"):
            ea = EventAccumulator(os.path.join(log_dir, file))
            ea.Reload()
            if tag in ea.Tags()['scalars']:
                vals = [e.value for e in ea.Scalars(tag)]
                if max(vals) > best_val: best_val = max(vals)
    return best_val

segmote_iou = get_best_metric('outputs/medical_segmentation/logs', 'Metrics/mIoU')
segmote_dice = get_best_metric('outputs/medical_segmentation/logs', 'Metrics/mDice')

baseline_iou = get_best_metric('outputs/moe_segmentation/logs', 'Metrics/mIoU')
baseline_dice = get_best_metric('outputs/moe_segmentation/logs', 'Metrics/mDice')

df = pd.DataFrame({
    'Model': ['MoE-Segformer (Baseline)', 'SegMoTE (Foundation Model)'],
    'BUSI Val mIoU': [f"{baseline_iou*100:.2f}%", f"{segmote_iou*100:.2f}%"],
    'BUSI Val mDice': [f"{baseline_dice*100:.2f}%", f"{segmote_dice*100:.2f}%"],
    'Total Parameters': ['8.07 Million', '94.20 Million']
})

print("\n================ BUSI DATASET COMPARISON RESULTS ================")
display(df)
print("================================================================")